# Notebook 02 - Feature Engineering for PyTorch

## Feature Engineering Overview

In this notebook, we prepare the cleaned modeling dataset for PyTorch models. The output is a set of numerical arrays, encoders, a scaler, metadata, and class weights.

The main goal is to convert the flight data into a format that neural networks can use without changing the target definition created in Notebook 01.


In [131]:
from pathlib import Path
import polars as pl
import pandas as pd
import numpy as np
import pickle
import json

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

## Data Loading

This section defines the project paths and points to the modeling parquet file created in the previous notebook.


In [132]:
BASE_DIR = Path.cwd().parent

GENERATED_DIR = BASE_DIR / "data" / "generated"
PROCESSED_DIR = GENERATED_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

model_base_path = GENERATED_DIR / "flights_2025_model_base.parquet"

print("Model base path:", model_base_path)
print("Exists:", model_base_path.exists())

Model base path: c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\flights_2025_model_base.parquet
Exists: True


We load the model base dataset and inspect its shape and schema before creating model-ready features.


In [133]:
df = pl.read_parquet(model_base_path)

print(df.shape)
df.head()

(6857418, 24)


YEAR,MONTH,DAY_OF_WEEK,flight_date,OP_UNIQUE_CARRIER,ORIGIN_AIRPORT_SEQ_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,DEST,DEST_CITY_NAME,DEST_STATE_ABR,ROUTE,CRS_DEP_TIME,CRS_DEP_HOUR,CRS_DEP_MINUTE,DEP_TIME_BLK,CRS_ARR_TIME,CRS_ARR_HOUR,CRS_ARR_MINUTE,ARR_TIME_BLK,DISTANCE,delay_class,delay_class_name
i64,i64,i64,date,str,i64,str,str,str,str,str,str,str,i64,i64,i64,str,i64,i64,i64,str,f64,i32,str
2025,11,1,2025-11-03,"""AA""",1014006,"""ABQ""","""Albuquerque, NM""","""NM""","""DFW""","""Dallas/Fort Worth, TX""","""TX""","""ABQ_DFW""",606,6,6,"""0600-0659""",904,9,4,"""0900-0959""",569.0,0,"""On time"""
2025,11,1,2025-11-03,"""AA""",1014006,"""ABQ""","""Albuquerque, NM""","""NM""","""DFW""","""Dallas/Fort Worth, TX""","""TX""","""ABQ_DFW""",820,8,20,"""0800-0859""",1118,11,18,"""1100-1159""",569.0,0,"""On time"""
2025,11,1,2025-11-03,"""AA""",1014006,"""ABQ""","""Albuquerque, NM""","""NM""","""DFW""","""Dallas/Fort Worth, TX""","""TX""","""ABQ_DFW""",1032,10,32,"""1000-1059""",1323,13,23,"""1300-1359""",569.0,0,"""On time"""
2025,11,1,2025-11-03,"""AA""",1014006,"""ABQ""","""Albuquerque, NM""","""NM""","""DFW""","""Dallas/Fort Worth, TX""","""TX""","""ABQ_DFW""",1212,12,12,"""1200-1259""",1502,15,2,"""1500-1559""",569.0,0,"""On time"""
2025,11,1,2025-11-03,"""AA""",1014006,"""ABQ""","""Albuquerque, NM""","""NM""","""DFW""","""Dallas/Fort Worth, TX""","""TX""","""ABQ_DFW""",1724,17,24,"""1700-1759""",2014,20,14,"""2000-2059""",569.0,0,"""On time"""


In [134]:
df.columns

['YEAR',
 'MONTH',
 'DAY_OF_WEEK',
 'flight_date',
 'OP_UNIQUE_CARRIER',
 'ORIGIN_AIRPORT_SEQ_ID',
 'ORIGIN',
 'ORIGIN_CITY_NAME',
 'ORIGIN_STATE_ABR',
 'DEST',
 'DEST_CITY_NAME',
 'DEST_STATE_ABR',
 'ROUTE',
 'CRS_DEP_TIME',
 'CRS_DEP_HOUR',
 'CRS_DEP_MINUTE',
 'DEP_TIME_BLK',
 'CRS_ARR_TIME',
 'CRS_ARR_HOUR',
 'CRS_ARR_MINUTE',
 'ARR_TIME_BLK',
 'DISTANCE',
 'delay_class',
 'delay_class_name']

In [135]:
df.schema

Schema([('YEAR', Int64),
        ('MONTH', Int64),
        ('DAY_OF_WEEK', Int64),
        ('flight_date', Date),
        ('OP_UNIQUE_CARRIER', String),
        ('ORIGIN_AIRPORT_SEQ_ID', Int64),
        ('ORIGIN', String),
        ('ORIGIN_CITY_NAME', String),
        ('ORIGIN_STATE_ABR', String),
        ('DEST', String),
        ('DEST_CITY_NAME', String),
        ('DEST_STATE_ABR', String),
        ('ROUTE', String),
        ('CRS_DEP_TIME', Int64),
        ('CRS_DEP_HOUR', Int64),
        ('CRS_DEP_MINUTE', Int64),
        ('DEP_TIME_BLK', String),
        ('CRS_ARR_TIME', Int64),
        ('CRS_ARR_HOUR', Int64),
        ('CRS_ARR_MINUTE', Int64),
        ('ARR_TIME_BLK', String),
        ('DISTANCE', Float64),
        ('delay_class', Int32),
        ('delay_class_name', String)])

Before changing the features, we check the target distribution again. This confirms that the 3-class target is still present after loading the saved file.


In [136]:
target_distribution = (
    df.group_by(["delay_class", "delay_class_name"])
      .agg(pl.len().alias("count"))
      .with_columns(
          ((pl.col("count") / df.height) * 100).round(4).alias("percentage")
      )
      .sort("delay_class")
)

target_distribution

delay_class,delay_class_name,count,percentage
i32,str,u32,f64
0,"""On time""",5402270,78.7799
1,"""Delay""",1116997,16.2889
2,"""Long delay""",338151,4.9312


In [137]:
target_distribution.write_csv(PROCESSED_DIR / "target_distribution_model_base.csv")

## Time-Based Feature Creation

Scheduled departure and arrival times are converted into minutes of the day. This makes the time values easier to use as numerical inputs.


In [138]:
df = df.with_columns([
    ((pl.col("CRS_DEP_HOUR") * 60) + pl.col("CRS_DEP_MINUTE")).alias("CRS_DEP_MINUTES_OF_DAY"),
    ((pl.col("CRS_ARR_HOUR") * 60) + pl.col("CRS_ARR_MINUTE")).alias("CRS_ARR_MINUTES_OF_DAY"),
])

Cyclic time features are created with sine and cosine transformations. This helps the model understand that times and days wrap around, such as late night being close to early morning.


In [139]:
df = df.with_columns([
    (2 * np.pi * pl.col("CRS_DEP_MINUTES_OF_DAY") / 1440).sin().alias("dep_time_sin"),
    (2 * np.pi * pl.col("CRS_DEP_MINUTES_OF_DAY") / 1440).cos().alias("dep_time_cos"),

    (2 * np.pi * pl.col("CRS_ARR_MINUTES_OF_DAY") / 1440).sin().alias("arr_time_sin"),
    (2 * np.pi * pl.col("CRS_ARR_MINUTES_OF_DAY") / 1440).cos().alias("arr_time_cos"),

    (2 * np.pi * pl.col("MONTH") / 12).sin().alias("month_sin"),
    (2 * np.pi * pl.col("MONTH") / 12).cos().alias("month_cos"),

    (2 * np.pi * pl.col("DAY_OF_WEEK") / 7).sin().alias("day_of_week_sin"),
    (2 * np.pi * pl.col("DAY_OF_WEEK") / 7).cos().alias("day_of_week_cos"),
])

In [140]:
df = df.with_columns(
    pl.when(pl.col("DAY_OF_WEEK").is_in([6, 7]))
      .then(1)
      .otherwise(0)
      .alias("is_weekend")
)

In [141]:
df.select([
    "MONTH",
    "DAY_OF_WEEK",
    "CRS_DEP_TIME",
    "CRS_DEP_MINUTES_OF_DAY",
    "dep_time_sin",
    "dep_time_cos",
    "CRS_ARR_TIME",
    "CRS_ARR_MINUTES_OF_DAY",
    "arr_time_sin",
    "arr_time_cos",
    "is_weekend",
    "delay_class"
]).head()

MONTH,DAY_OF_WEEK,CRS_DEP_TIME,CRS_DEP_MINUTES_OF_DAY,dep_time_sin,dep_time_cos,CRS_ARR_TIME,CRS_ARR_MINUTES_OF_DAY,arr_time_sin,arr_time_cos,is_weekend,delay_class
i64,i64,i64,i64,f64,f64,i64,i64,f64,f64,i32,i32
11,1,606,366,0.999657,-0.026177,904,544,0.694658,-0.71934,0,0
11,1,820,500,0.819152,-0.573576,1118,678,0.182236,-0.983255,0,0
11,1,1032,632,0.374607,-0.927184,1323,803,-0.354291,-0.935135,0,0
11,1,1212,732,-0.052336,-0.99863,1502,902,-0.71325,-0.700909,0,0
11,1,1724,1044,-0.987688,-0.156434,2014,1214,-0.833886,0.551937,0,0


## Categorical and Numerical Feature Selection

The selected categorical features describe airlines, airports, routes, and scheduled time blocks. The numerical features include distance and engineered time variables.


In [142]:
categorical_cols = [
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "ORIGIN_STATE_ABR",
    "DEST",
    "DEST_STATE_ABR",
    "ROUTE",
    "DEP_TIME_BLK",
    "ARR_TIME_BLK",
]

In [143]:
numeric_cols = [
    "DISTANCE",
    "CRS_DEP_MINUTES_OF_DAY",
    "CRS_ARR_MINUTES_OF_DAY",
    "dep_time_sin",
    "dep_time_cos",
    "arr_time_sin",
    "arr_time_cos",
    "month_sin",
    "month_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "is_weekend",
]

In [144]:
target_col = "delay_class"

We check that all required columns exist and then remove rows with missing values in the selected model features.


In [145]:
missing_categorical = [col for col in categorical_cols if col not in df.columns]
missing_numeric = [col for col in numeric_cols if col not in df.columns]

print("Missing categorical columns:", missing_categorical)
print("Missing numeric columns:", missing_numeric)

Missing categorical columns: []
Missing numeric columns: []


In [146]:
required_cols = categorical_cols + numeric_cols + [target_col, "MONTH"]

null_check = df.select([
    pl.col(col).is_null().sum().alias(col)
    for col in required_cols
])

null_check

OP_UNIQUE_CARRIER,ORIGIN,ORIGIN_STATE_ABR,DEST,DEST_STATE_ABR,ROUTE,DEP_TIME_BLK,ARR_TIME_BLK,DISTANCE,CRS_DEP_MINUTES_OF_DAY,CRS_ARR_MINUTES_OF_DAY,dep_time_sin,dep_time_cos,arr_time_sin,arr_time_cos,month_sin,month_cos,day_of_week_sin,day_of_week_cos,is_weekend,delay_class,MONTH
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [147]:
df_model = df.drop_nulls(required_cols)

print("Before:", df.shape)
print("After:", df_model.shape)
print("Removed:", df.height - df_model.height)

Before: (6857418, 35)
After: (6857418, 35)
Removed: 0


## Temporal Train/Validation/Test Split

The split is based on time instead of random sampling:

- Train: months 1-9.
- Validation: month 10.
- Test: months 11-12.

This is more realistic for flight delay prediction because future months should be evaluated using patterns learned from earlier months.


In [148]:
train_df = df_model.filter(pl.col("MONTH").is_between(1, 9))
val_df = df_model.filter(pl.col("MONTH") == 10)
test_df = df_model.filter(pl.col("MONTH").is_between(11, 12))

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (5128628, 35)
Validation: (601570, 35)
Test: (1127220, 35)


In [149]:
def get_split_distribution(data, split_name):
    return (
        data.group_by(target_col)
            .agg(pl.len().alias("count"))
            .with_columns([
                ((pl.col("count") / data.height) * 100).round(4).alias("percentage"),
                pl.lit(split_name).alias("split")
            ])
            .sort(target_col)
    )

split_distribution = pl.concat([
    get_split_distribution(train_df, "train"),
    get_split_distribution(val_df, "validation"),
    get_split_distribution(test_df, "test"),
])

split_distribution

delay_class,count,percentage,split
i32,u32,f64,str
0,4050886,78.9858,"""train"""
1,823789,16.0626,"""train"""
2,253953,4.9517,"""train"""
0,483672,80.4016,"""validation"""
1,95899,15.9415,"""validation"""
2,21999,3.6569,"""validation"""
0,867712,76.9781,"""test"""
1,197309,17.504,"""test"""
2,62199,5.5179,"""test"""


The split keeps the same general class imbalance across train, validation, and test. This is expected because long delays are rare in the full dataset.


In [150]:
split_distribution.write_csv(PROCESSED_DIR / "split_class_distribution.csv")

## Encoding Categorical Variables

The split data is converted to pandas so categorical values can be encoded as integer IDs. These IDs are later passed into embedding layers in the neural networks.


In [151]:
train_pd = train_df.select(categorical_cols + numeric_cols + [target_col]).to_pandas()
val_pd = val_df.select(categorical_cols + numeric_cols + [target_col]).to_pandas()
test_pd = test_df.select(categorical_cols + numeric_cols + [target_col]).to_pandas()

print(train_pd.shape)
print(val_pd.shape)
print(test_pd.shape)

(5128628, 21)
(601570, 21)
(1127220, 21)


Encoders are fitted only on the training data. Unknown categories in validation or test are mapped to 0, which is reserved for unseen values.


In [152]:
category_encoders = {}
category_cardinalities = {}

for col in categorical_cols:
    unique_values = train_pd[col].astype(str).unique()
    
    encoder = {value: idx + 1 for idx, value in enumerate(unique_values)}
    
    category_encoders[col] = encoder
    category_cardinalities[col] = len(encoder) + 1  # +1 for unknown category
    
    train_pd[col] = train_pd[col].astype(str).map(encoder).fillna(0).astype("int64")
    val_pd[col] = val_pd[col].astype(str).map(encoder).fillna(0).astype("int64")
    test_pd[col] = test_pd[col].astype(str).map(encoder).fillna(0).astype("int64")

category_cardinalities

{'OP_UNIQUE_CARRIER': 15,
 'ORIGIN': 350,
 'ORIGIN_STATE_ABR': 53,
 'DEST': 350,
 'DEST_STATE_ABR': 53,
 'ROUTE': 6688,
 'DEP_TIME_BLK': 20,
 'ARR_TIME_BLK': 20}

In [153]:
with open(PROCESSED_DIR / "category_cardinalities.json", "w") as f:
    json.dump(category_cardinalities, f, indent=4)

with open(PROCESSED_DIR / "category_encoders.pkl", "wb") as f:
    pickle.dump(category_encoders, f)

## Scaling Numerical Variables

Numerical features are scaled with `StandardScaler`. The scaler is fitted on the training set and then applied to validation and test sets to avoid data leakage.


In [154]:
scaler = StandardScaler()

train_pd[numeric_cols] = scaler.fit_transform(train_pd[numeric_cols])
val_pd[numeric_cols] = scaler.transform(val_pd[numeric_cols])
test_pd[numeric_cols] = scaler.transform(test_pd[numeric_cols])

In [155]:
with open(PROCESSED_DIR / "numeric_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [156]:
train_pd[numeric_cols].describe()

,DISTANCE,CRS_DEP_MINUTES_OF_DAY,CRS_ARR_MINUTES_OF_DAY,dep_time_sin,dep_time_cos,arr_time_sin,arr_time_cos,month_sin,month_cos,day_of_week_sin,day_of_week_cos,is_weekend
count,5.128628e+06,5.128628e+06,5.128628e+06,5.128628e+06,5.128628e+06,5.128628e+06,5.128628e+06,5.128628e+06,5.128628e+06,5.128628e+06,5.128628e+06,5.128628e+06
mean,4.600783e-17,-4.117955e-17,-8.196911e-17,-4.420398e-17,-6.800619e-16,2.408137e-16,1.525935e-16,-2.644536e-15,-2.740076e-16,-4.285149e-16,5.359784e-16,-8.334832e-18
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-1.350856e+00,-2.709699e+00,-2.912261e+00,-1.199604e+00,-1.164207e+00,-1.138531e+00,-1.139876e+00,-1.379977e+00,-1.089210e+00,-1.396898e+00,-1.248735e+00,-6.046685e-01
25%,-7.341074e-01,-8.809178e-01,-7.864201e-01,-9.888741e-01,-9.226726e-01,-9.156886e-01,-9.266310e-01,-7.344268e-01,-1.089210e+00,-1.122974e+00,-1.248735e+00,-6.046685e-01
50%,-2.533085e-01,-3.063589e-02,3.046895e-02,-2.305522e-01,-2.094326e-01,-2.731012e-01,-2.871240e-01,5.566730e-01,3.882997e-01,-1.387966e-02,-2.918817e-01,-6.046685e-01
75%,3.949410e-01,8.398908e-01,8.312775e-01,1.076472e+00,6.096380e-01,7.556970e-01,9.188033e-01,1.029248e+00,3.882997e-01,1.095215e+00,9.012947e-01,1.653799e+00
max,7.044887e+00,2.145681e+00,1.712488e+00,1.491568e+00,2.383483e+00,2.031485e+00,1.754322e+00,1.202223e+00,1.865809e+00,1.369139e+00,1.432308e+00,1.653799e+00


The scaled training features have means close to 0 and standard deviations close to 1. This helps neural network optimization behave more consistently.


The final feature matrices separate categorical IDs from scaled numerical features. This matches the input format used by the PyTorch datasets in the modeling notebooks.


In [157]:
X_train_cat = train_pd[categorical_cols].values.astype("int64")
X_val_cat = val_pd[categorical_cols].values.astype("int64")
X_test_cat = test_pd[categorical_cols].values.astype("int64")

X_train_num = train_pd[numeric_cols].values.astype("float32")
X_val_num = val_pd[numeric_cols].values.astype("float32")
X_test_num = test_pd[numeric_cols].values.astype("float32")

y_train = train_pd[target_col].values.astype("int64")
y_val = val_pd[target_col].values.astype("int64")
y_test = test_pd[target_col].values.astype("int64")

In [158]:
print("X_train_cat:", X_train_cat.shape)
print("X_train_num:", X_train_num.shape)
print("y_train:", y_train.shape)

print("X_val_cat:", X_val_cat.shape)
print("X_val_num:", X_val_num.shape)
print("y_val:", y_val.shape)

print("X_test_cat:", X_test_cat.shape)
print("X_test_num:", X_test_num.shape)
print("y_test:", y_test.shape)

X_train_cat: (5128628, 8)
X_train_num: (5128628, 12)
y_train: (5128628,)
X_val_cat: (601570, 8)
X_val_num: (601570, 12)
y_val: (601570,)
X_test_cat: (1127220, 8)
X_test_num: (1127220, 12)
y_test: (1127220,)


## Class Weight Calculation

Class weights are calculated from the training target distribution. They reduce the effect of class imbalance by giving more weight to the less frequent classes.


In [159]:
classes = np.array(sorted(train_pd[target_col].unique()))

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights_dict = {
    int(cls): float(weight)
    for cls, weight in zip(classes, class_weights)
}

class_weights_dict

{0: 0.42201697768504637, 1: 2.0752190993891233, 2: 6.731728574447502}

The highest weight is assigned to the Long delay class because it has the fewest examples. These weights are saved and reused during model training.


In [160]:
with open(PROCESSED_DIR / "class_weights.json", "w") as f:
    json.dump(class_weights_dict, f, indent=4)

## Saving PyTorch-Ready Datasets

The arrays are saved as compressed `.npz` files. These files are smaller and faster to load than the full dataframe during model training.


In [161]:
np.savez_compressed(
    PROCESSED_DIR / "train_data.npz",
    X_cat=X_train_cat,
    X_num=X_train_num,
    y=y_train
)

np.savez_compressed(
    PROCESSED_DIR / "val_data.npz",
    X_cat=X_val_cat,
    X_num=X_val_num,
    y=y_val
)

np.savez_compressed(
    PROCESSED_DIR / "test_data.npz",
    X_cat=X_test_cat,
    X_num=X_test_num,
    y=y_test
)

Metadata is saved with the feature names, class names, category sizes, and split strategy. This keeps the modeling notebooks consistent with the preprocessing step.


In [162]:
metadata = {
    "categorical_cols": categorical_cols,
    "numeric_cols": numeric_cols,
    "target_col": target_col,
    "num_classes": 3,
    "class_names": {
        "0": "On time",
        "1": "Delay",
        "2": "Long delay"
    },
    "category_cardinalities": category_cardinalities,
    "split_strategy": {
        "train": "Months 1-9",
        "validation": "Month 10",
        "test": "Months 11-12"
    }
}

with open(PROCESSED_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

metadata

{'categorical_cols': ['OP_UNIQUE_CARRIER',
  'ORIGIN',
  'ORIGIN_STATE_ABR',
  'DEST',
  'DEST_STATE_ABR',
  'ROUTE',
  'DEP_TIME_BLK',
  'ARR_TIME_BLK'],
 'numeric_cols': ['DISTANCE',
  'CRS_DEP_MINUTES_OF_DAY',
  'CRS_ARR_MINUTES_OF_DAY',
  'dep_time_sin',
  'dep_time_cos',
  'arr_time_sin',
  'arr_time_cos',
  'month_sin',
  'month_cos',
  'day_of_week_sin',
  'day_of_week_cos',
  'is_weekend'],
 'target_col': 'delay_class',
 'num_classes': 3,
 'class_names': {'0': 'On time', '1': 'Delay', '2': 'Long delay'},
 'category_cardinalities': {'OP_UNIQUE_CARRIER': 15,
  'ORIGIN': 350,
  'ORIGIN_STATE_ABR': 53,
  'DEST': 350,
  'DEST_STATE_ABR': 53,
  'ROUTE': 6688,
  'DEP_TIME_BLK': 20,
  'ARR_TIME_BLK': 20},
 'split_strategy': {'train': 'Months 1-9',
  'validation': 'Month 10',
  'test': 'Months 11-12'}}

In [163]:
print("Notebook 02 completed")

print("\nProcessed files:")
print(PROCESSED_DIR / "train_data.npz")
print(PROCESSED_DIR / "val_data.npz")
print(PROCESSED_DIR / "test_data.npz")
print(PROCESSED_DIR / "metadata.json")
print(PROCESSED_DIR / "category_cardinalities.json")
print(PROCESSED_DIR / "category_encoders.pkl")
print(PROCESSED_DIR / "numeric_scaler.pkl")
print(PROCESSED_DIR / "class_weights.json")

print("\nFinal shapes:")
print("Train categorical:", X_train_cat.shape)
print("Train numerical:", X_train_num.shape)
print("Train target:", y_train.shape)

print("Validation categorical:", X_val_cat.shape)
print("Validation numerical:", X_val_num.shape)
print("Validation target:", y_val.shape)

print("Test categorical:", X_test_cat.shape)
print("Test numerical:", X_test_num.shape)
print("Test target:", y_test.shape)

Notebook 02 completed

Processed files:
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\train_data.npz
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\val_data.npz
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\test_data.npz
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\metadata.json
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\category_cardinalities.json
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\category_encoders.pkl
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\numeric_scaler.pkl
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\cla

## Notebook 02 Conclusion

This notebook created the PyTorch-ready train, validation, and test files. It also saved the categorical encoders, numerical scaler, metadata, and class weights needed by the model notebooks.
